In [1]:
from pathlib import Path
import json, hashlib, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve)
import sklearn

pd.set_option("display.max_columns", 40)
print("Python:", platform.python_version(), "pandas:", pd.__version__, "sklearn:", sklearn.__version__)

INPUT_DIR = Path("ablation_prepared")
OUTPUT_DIR = Path("ablation_results")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)
CONTRACT_FILE = INPUT_DIR / "modeling_contract.json"
if not CONTRACT_FILE.exists():
    raise FileNotFoundError(f"Run Notebook 1 first; missing {CONTRACT_FILE.resolve()}")
contract = json.loads(CONTRACT_FILE.read_text(encoding="utf-8"))
print("Notebook 1 source:", contract["source_file"])
print("Horizon unit:", contract["forecast_horizon_unit"])

Python: 3.11.9 pandas: 3.0.1 sklearn: 1.9.0
Notebook 1 source: Miami_and_Hialeah_with_environmental_factors (2).csv
Horizon unit: indexed calendar-month reporting periods, NOT exact 7-day weeks


In [13]:
CITIES = ("Miami", "Hialeah")
HORIZONS = (1, 2, 4)
TEST_FRACTION = 0.20
PEAK_QUANTILE = 0.80
DECISION_THRESHOLD = 0.50
RANDOM_SEED = 42
MIN_TRAIN_ROWS = 35
MIN_TEST_ROWS = 12
REQUIRE_BOTH_TEST_CLASSES = False  # undefined ROC-AUC otherwise; output NaN and warn

HISTORY_FEATURES = contract["history_features"]
ENVIRONMENT_FEATURES = contract["environment_features"]
FEATURE_SETS = {
    "Environmental": ENVIRONMENT_FEATURES,
    "History": HISTORY_FEATURES,
    "Hybrid": ENVIRONMENT_FEATURES + HISTORY_FEATURES,
}
assert set(HISTORY_FEATURES).isdisjoint(ENVIRONMENT_FEATURES)
assert sorted(contract["horizons"]) == list(HORIZONS)
assert "NOT exact 7-day weeks" in contract["forecast_horizon_unit"]
print({k: len(v) for k, v in FEATURE_SETS.items()})

{'Environmental': 48, 'History': 7, 'Hybrid': 55}


In [14]:
def make_model(algorithm: str):
    if algorithm == "LogisticRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                class_weight="balanced", max_iter=3000, solver="lbfgs",
                random_state=RANDOM_SEED)),
        ])
    if algorithm == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", RandomForestClassifier(
                n_estimators=400, max_features="sqrt", min_samples_leaf=5,
                class_weight="balanced_subsample", random_state=RANDOM_SEED,
                n_jobs=-1)),
        ])
    raise ValueError(algorithm)

ALGORITHMS = ("LogisticRegression", "RandomForest")

In [4]:
def read_prepared(h):
    path = INPUT_DIR / f"modeling_h{h}_periods.csv"
    if not path.exists():
        raise FileNotFoundError(path.resolve())
    d = pd.read_csv(path)
    expected = ["City", "YEAR", "MONTH", "WEEK", "period_index", "period_start", "period_end",
                "mosquito_count", "n_active_traps", "horizon_periods",
                f"target_count_h{h}", f"target_index_h{h}", f"target_start_h{h}", f"target_end_h{h}"]
    absent = sorted(set(expected + ENVIRONMENT_FEATURES + HISTORY_FEATURES) - set(d.columns))
    if absent:
        raise ValueError(f"h={h} missing columns: {absent}")
    for c in ["period_start", "period_end", f"target_start_h{h}", f"target_end_h{h}"]:
        d[c] = pd.to_datetime(d[c], errors="raise")
    d["City"] = d["City"].astype(str).str.strip()
    numeric_columns = list(dict.fromkeys(["period_index", "mosquito_count", "n_active_traps",
         f"target_count_h{h}", f"target_index_h{h}"] + ENVIRONMENT_FEATURES + HISTORY_FEATURES))
    for c in numeric_columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    if d[expected].isna().any().any():
        raise ValueError(f"h={h}: missing required date/index/outcome values")
    if d.duplicated(["City", "period_index"]).any():
        raise ValueError(f"h={h}: duplicate city-origin indices")
    if not d.horizon_periods.eq(h).all():
        raise ValueError(f"h={h}: unexpected horizon labels")
    if not (d[f"target_index_h{h}"] - d.period_index).eq(h).all():
        raise ValueError(f"h={h}: not a genuine h-period target")
    if not (d[f"target_start_h{h}"] > d.period_end).all():
        raise ValueError(f"h={h}: target starts before/equal to origin end")
    if (d[["mosquito_count", f"target_count_h{h}"]] < 0).any().any():
        raise ValueError(f"h={h}: negative observed mosquito counts")
    if not d["City"].isin(CITIES).all():
        raise ValueError(f"h={h}: unexpected city names")
    return d.sort_values(["City", "period_index"]).reset_index(drop=True)

prepared_files = {h: read_prepared(h) for h in HORIZONS}
print("Rows by horizon and city:")
for h, d in prepared_files.items():
    print(f"h={h}", d.groupby("City").size().to_dict())

Rows by horizon and city:
h=1 {'Hialeah': 112, 'Miami': 207}
h=2 {'Hialeah': 115, 'Miami': 207}
h=4 {'Hialeah': 115, 'Miami': 205}


In [5]:
def chronological_split(city_df, h):
    d = city_df.sort_values("period_index").reset_index(drop=True)
    n = len(d)
    if n < MIN_TRAIN_ROWS + MIN_TEST_ROWS:
        warnings.warn(f"Only {n} eligible rows (h={h}, city={d.City.iloc[0]}); results may be unstable")
    cut = int(np.floor(n * (1 - TEST_FRACTION)))
    if cut < 2 or cut >= n:
        raise ValueError("Not enough rows for chronological split")
    first_test_origin_index = int(d.iloc[cut]["period_index"])
    # Require training target period to be strictly earlier than first test origin period.
    train = d.iloc[:cut].loc[d.iloc[:cut][f"target_index_h{h}"] < first_test_origin_index].copy()
    test = d.iloc[cut:].copy()
    if not len(train) or not len(test):
        raise ValueError("Empty training or test subset after purge")
    if not (train[f"target_end_h{h}"].max() < test["period_start"].min()):
        raise AssertionError("Training outcome overlaps or follows test origin")
    if not (train.period_index.max() < test.period_index.min()):
        raise AssertionError("Train/test origin overlap")
    # Training-only definition of elevated mosquito activity.
    peak_cutoff = float(np.quantile(train["mosquito_count"].to_numpy(), PEAK_QUANTILE))
    train["peak_target"] = (train[f"target_count_h{h}"] >= peak_cutoff).astype(int)
    test["peak_target"] = (test[f"target_count_h{h}"] >= peak_cutoff).astype(int)
    split_info = {
        "n_eligible": n, "n_train_before_purge": cut, "n_train": len(train),
        "n_purged": cut - len(train), "n_test": len(test),
        "first_test_origin": test["period_start"].min().date().isoformat(),
        "last_train_target_end": train[f"target_end_h{h}"].max().date().isoformat(),
        "peak_threshold_train_80th_origin_count": peak_cutoff,
        "train_prevalence": float(train.peak_target.mean()),
        "test_prevalence": float(test.peak_target.mean()),
        "train_positive": int(train.peak_target.sum()),
        "test_positive": int(test.peak_target.sum()),
    }
    return train, test, split_info

In [6]:
def select_train_usable_features(train, feature_names):
    included, removed = [], []
    for c in feature_names:
        values = pd.to_numeric(train[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        usable = values.dropna()
        if len(usable) == 0:
            removed.append({"feature": c, "reason": "all missing in training"})
        elif usable.nunique() <= 1:
            removed.append({"feature": c, "reason": "constant in training"})
        else:
            included.append(c)
    if not included:
        raise ValueError("No usable predictors for this feature set")
    return included, removed


def model_matrix(df, included):
    x = df[included].copy()
    return x.replace([np.inf, -np.inf], np.nan)

In [7]:
def score_predictions(y_true, probabilities, threshold=DECISION_THRESHOLD):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    if not np.isfinite(p).all() or (p < 0).any() or (p > 1).any():
        raise ValueError("Invalid predicted probabilities")
    predicted = (p >= threshold).astype(int)
    cm = confusion_matrix(y, predicted, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    ap = float(average_precision_score(y, p)) if y.sum() else np.nan
    auc = float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan
    return {
        "ROC_AUC": auc, "Average_Precision": ap,
        "Accuracy": float(accuracy_score(y, predicted)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, predicted)) if len(np.unique(y)) == 2 else np.nan,
        "Precision": float(precision_score(y, predicted, zero_division=0)),
        "Recall": float(recall_score(y, predicted, zero_division=0)),
        "F1": float(f1_score(y, predicted, zero_division=0)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "n_test": int(len(y)), "n_test_positive": int(y.sum()),
        "test_positive_fraction": float(y.mean()),
        "no_skill_AP_baseline": float(y.mean()),
        "probability_threshold": float(threshold),
    }

In [8]:
metrics_records = []
prediction_frames = []
split_records = []
feature_audit_records = []
trained_models = {}

for h, data in prepared_files.items():
    for city in CITIES:
        city_data = data.loc[data.City == city].copy()
        if city_data.empty:
            warnings.warn(f"No rows for city={city} horizon={h}; skipping")
            continue
        train, test, info = chronological_split(city_data, h)
        info.update({"City": city, "horizon_periods": h})
        split_records.append(info)
        print(f"{city}, h={h}: train={len(train)}, test={len(test)}, "
              f"cutoff={info['peak_threshold_train_80th_origin_count']:.2f}, "
              f"test-positive={int(test.peak_target.sum())}/{len(test)}")
        if train.peak_target.nunique() < 2:
            warnings.warn(f"Only one training class: {city}, horizon={h}. Cannot fit classifiers; skipping.")
            continue
        if test.peak_target.nunique() < 2:
            warnings.warn(f"Only one test class: {city}, horizon={h}; ROC-AUC undefined.")
            if REQUIRE_BOTH_TEST_CLASSES:
                continue

        for feature_set, proposed in FEATURE_SETS.items():
            columns, removed = select_train_usable_features(train, proposed)
            for item in removed:
                feature_audit_records.append({"City": city, "horizon_periods": h,
                    "feature_set": feature_set, "feature": item["feature"], "reason": item["reason"]})
            x_train = model_matrix(train, columns)
            x_test = model_matrix(test, columns)
            y_train = train.peak_target.to_numpy(dtype=int)
            y_test = test.peak_target.to_numpy(dtype=int)
            for algorithm in ALGORITHMS:
                model = make_model(algorithm)
                model.fit(x_train, y_train)
                probs = model.predict_proba(x_test)[:, 1]
                scored = score_predictions(y_test, probs)
                scored.update({"City": city, "horizon_periods": h,
                    "Algorithm": algorithm, "Feature_Set": feature_set,
                    "n_train": len(train), "n_features": len(columns),
                    "peak_cutoff": info["peak_threshold_train_80th_origin_count"]})
                metrics_records.append(scored)
                trained_models[(city, h, algorithm, feature_set)] = (model, columns)
                frame = test[["City", "YEAR", "MONTH", "WEEK", "period_index",
                    "period_start", "period_end", f"target_start_h{h}",
                    f"target_end_h{h}", f"target_index_h{h}",
                    f"target_count_h{h}", "peak_target"]].copy()
                frame = frame.rename(columns={f"target_start_h{h}": "target_start",
                    f"target_end_h{h}": "target_end",
                    f"target_index_h{h}": "target_index",
                    f"target_count_h{h}": "observed_future_count",
                    "peak_target": "observed_peak_label"})
                frame["horizon_periods"] = h
                frame["Algorithm"] = algorithm
                frame["Feature_Set"] = feature_set
                frame["predicted_probability"] = probs
                frame["predicted_peak_label"] = (probs >= DECISION_THRESHOLD).astype(int)
                frame["peak_cutoff_train"] = info["peak_threshold_train_80th_origin_count"]
                prediction_frames.append(frame)

if not metrics_records:
    raise RuntimeError("No valid model fits. Review class counts and data coverage.")
metrics = pd.DataFrame(metrics_records).sort_values(["City", "horizon_periods", "Algorithm", "Feature_Set"])
predictions = pd.concat(prediction_frames, ignore_index=True)
splits = pd.DataFrame(split_records)
feature_audit = pd.DataFrame(feature_audit_records,
    columns=["City", "horizon_periods", "feature_set", "feature", "reason"])
metrics.to_csv(OUTPUT_DIR / "ablation_performance_all.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "ablation_predictions_all.csv", index=False)
splits.to_csv(OUTPUT_DIR / "chronological_split_audit.csv", index=False)
feature_audit.to_csv(OUTPUT_DIR / "excluded_training_features.csv", index=False)
print(f"Fitted {len(metrics)} models. Output directory: {OUTPUT_DIR.resolve()}")
display(metrics[["City", "horizon_periods", "Algorithm", "Feature_Set", "ROC_AUC",
                 "Average_Precision", "F1", "n_train", "n_test", "n_test_positive"]])

Miami, h=1: train=164, test=42, cutoff=1806.20, test-positive=12/42
Hialeah, h=1: train=88, test=23, cutoff=303.80, test-positive=16/23
Miami, h=2: train=163, test=42, cutoff=1812.80, test-positive=13/42
Hialeah, h=2: train=91, test=23, cutoff=305.00, test-positive=16/23
Miami, h=4: train=160, test=41, cutoff=1871.80, test-positive=12/41
Hialeah, h=4: train=90, test=23, cutoff=328.20, test-positive=13/23
Fitted 36 models. Output directory: C:\Users\lamby\Downloads\Final Code 2\Final Code\ablation_results


,City,horizon_periods,Algorithm,Feature_Set,ROC_AUC,Average_Precision,F1,n_train,n_test,n_test_positive
6,Hialeah,1,LogisticRegression,Environmental,0.857143,0.945747,0.476190,88,23,16
8,Hialeah,1,LogisticRegression,History,0.875000,0.927792,0.941176,88,23,16
10,Hialeah,1,LogisticRegression,Hybrid,0.910714,0.963645,0.814815,88,23,16
7,Hialeah,1,RandomForest,Environmental,0.964286,0.983786,0.769231,88,23,16
9,Hialeah,1,RandomForest,History,0.901786,0.938153,0.969697,88,23,16
11,Hialeah,1,RandomForest,Hybrid,0.964286,0.983786,0.827586,88,23,16
18,Hialeah,2,LogisticRegression,Environmental,0.803571,0.905372,0.400000,91,23,16
20,Hialeah,2,LogisticRegression,History,0.839286,0.868417,0.941176,91,23,16
22,Hialeah,2,LogisticRegression,Hybrid,0.821429,0.860428,0.875000,91,23,16
19,Hialeah,2,RandomForest,Environmental,0.919643,0.954894,0.608696,91,23,16


In [9]:
audit_rows=[]
for (city,h,algorithm,feature_set),g in predictions.groupby(
        ["City","horizon_periods","Algorithm","Feature_Set"]):
    computed=score_predictions(g.observed_peak_label.to_numpy(),g.predicted_probability.to_numpy())
    row=metrics.loc[(metrics.City==city)&(metrics.horizon_periods==h)&
        (metrics.Algorithm==algorithm)&(metrics.Feature_Set==feature_set)].iloc[0]
    for metric in ["ROC_AUC","Average_Precision","F1","Precision","Recall"]:
        a,b=computed[metric],row[metric]
        if not (pd.isna(a) and pd.isna(b)):
            assert np.isclose(a,b,rtol=0,atol=1e-10),f"Recompute mismatch {metric} {city} {h}"
    audit_rows.append({"City":city,"horizon_periods":h,"Algorithm":algorithm,
        "Feature_Set":feature_set,"verified":True})
# Three matched ablations per city/horizon/algorithm must use same targets and origins.
for (city,h,algorithm),g in predictions.groupby(["City","horizon_periods","Algorithm"]):
    sub=[]
    for fs,t in g.groupby("Feature_Set"):
        sub.append(t.sort_values("period_index")[["period_index","target_index", "observed_future_count",
            "observed_peak_label"]].reset_index(drop=True))
    if len(sub)!=3 or not all(sub[0].equals(other) for other in sub[1:]):
        raise AssertionError(f"Non-matched ablation samples for {city} h={h} {algorithm}")
print(f"Recomputed all {len(audit_rows)} model evaluations and confirmed matched ablation observations.")

Recomputed all 36 model evaluations and confirmed matched ablation observations.


In [10]:
COLOR_MAP = {"Environmental":"#3977b9", "History":"#e28230", "Hybrid":"#369657"}
for (city,h,algorithm),g in predictions.groupby(["City","horizon_periods","Algorithm"]):
    fig, axes=plt.subplots(1,2,figsize=(12,4.8),dpi=160)
    prevalence=float(g.observed_peak_label.mean())
    for fs in ("Environmental", "History", "Hybrid"):
        s=g.loc[g.Feature_Set==fs].sort_values("period_index")
        if s.empty: continue
        y=s.observed_peak_label.to_numpy(dtype=int)
        p=s.predicted_probability.to_numpy(dtype=float)
        auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan
        ap=average_precision_score(y,p) if y.sum() else np.nan
        if np.isfinite(auc):
            fpr,tpr,_=roc_curve(y,p)
            axes[0].plot(fpr,tpr,lw=1.8,color=COLOR_MAP[fs],label=f"{fs} (AUC={auc:.3f})")
        else:
            axes[0].plot([],[],color=COLOR_MAP[fs],label=f"{fs} (AUC undefined)")
        if np.isfinite(ap):
            prec,rec,_=precision_recall_curve(y,p)
            axes[1].step(rec,prec,where="post",lw=1.7,color=COLOR_MAP[fs],
                         label=f"{fs} (AP={ap:.3f})")
        else:
            axes[1].plot([],[],color=COLOR_MAP[fs],label=f"{fs} (AP undefined)")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=0.65,label="Random ranking")
    axes[1].axhline(prevalence,color="black",ls="--",lw=1,alpha=0.75,
                    label=f"No-skill AP={prevalence:.3f}")
    axes[0].set(xlim=(0,1),ylim=(0,1.02),xlabel="False positive rate",
                ylabel="True positive rate",title="ROC curve")
    axes[1].set(xlim=(0,1),ylim=(0,1.02),xlabel="Recall",ylabel="Precision",
                title="Precision–recall curve")
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8,loc="best")
    fig.suptitle(f"{city} | {h} indexed period(s) ahead | {algorithm}",fontsize=12)
    fig.tight_layout()
    base=FIG_DIR/f"curves_{city}_h{h}_{algorithm}"
    fig.savefig(base.with_suffix(".png"),bbox_inches="tight",dpi=250)
    fig.savefig(base.with_suffix(".pdf"),bbox_inches="tight")
    plt.close(fig)
print(f"Saved ROC/PR figures in {FIG_DIR.resolve()}")

Saved ROC/PR figures in C:\Users\lamby\Downloads\Final Code 2\Final Code\ablation_results\figures


In [11]:
main_cols=["City","horizon_periods","Algorithm","Feature_Set", "ROC_AUC",
    "Average_Precision","Precision","Recall","F1","n_test", "n_test_positive",
    "test_positive_fraction","peak_cutoff"]
publication_table=metrics[main_cols].copy()
publication_table.to_csv(OUTPUT_DIR/"publication_ablation_table.csv",index=False)
deltas=[]
for (city,h,algorithm),s in metrics.groupby(["City","horizon_periods","Algorithm"]):
    byset=s.set_index("Feature_Set")
    if not set(FEATURE_SETS).issubset(byset.index): continue
    record={"City":city,"horizon_periods":h,"Algorithm":algorithm}
    for metric in ["ROC_AUC","Average_Precision","F1"]:
        record[f"Hybrid_minus_History_{metric}"]=float(byset.loc["Hybrid",metric]-byset.loc["History",metric])
        record[f"Hybrid_minus_Environmental_{metric}"]=float(byset.loc["Hybrid",metric]-byset.loc["Environmental",metric])
    deltas.append(record)
deltas_df=pd.DataFrame(deltas)
deltas_df.to_csv(OUTPUT_DIR/"ablation_descriptive_deltas.csv",index=False)
display(publication_table.round(3))
print("Descriptive hybrid-feature gains (can be negative):")
display(deltas_df.round(3))

,City,horizon_periods,Algorithm,Feature_Set,ROC_AUC,Average_Precision,Precision,Recall,F1,n_test,n_test_positive,test_positive_fraction,peak_cutoff
6,Hialeah,1,LogisticRegression,Environmental,0.857,0.946,1.000,0.312,0.476,23,16,0.696,303.8
8,Hialeah,1,LogisticRegression,History,0.875,0.928,0.889,1.000,0.941,23,16,0.696,303.8
10,Hialeah,1,LogisticRegression,Hybrid,0.911,0.964,1.000,0.688,0.815,23,16,0.696,303.8
7,Hialeah,1,RandomForest,Environmental,0.964,0.984,1.000,0.625,0.769,23,16,0.696,303.8
9,Hialeah,1,RandomForest,History,0.902,0.938,0.941,1.000,0.970,23,16,0.696,303.8
11,Hialeah,1,RandomForest,Hybrid,0.964,0.984,0.923,0.750,0.828,23,16,0.696,303.8
18,Hialeah,2,LogisticRegression,Environmental,0.804,0.905,1.000,0.250,0.400,23,16,0.696,305.0
20,Hialeah,2,LogisticRegression,History,0.839,0.868,0.889,1.000,0.941,23,16,0.696,305.0
22,Hialeah,2,LogisticRegression,Hybrid,0.821,0.860,0.875,0.875,0.875,23,16,0.696,305.0
19,Hialeah,2,RandomForest,Environmental,0.920,0.955,1.000,0.438,0.609,23,16,0.696,305.0


Descriptive hybrid-feature gains (can be negative):


,City,horizon_periods,Algorithm,Hybrid_minus_History_ROC_AUC,Hybrid_minus_Environmental_ROC_AUC,Hybrid_minus_History_Average_Precision,Hybrid_minus_Environmental_Average_Precision,Hybrid_minus_History_F1,Hybrid_minus_Environmental_F1
0,Hialeah,1,LogisticRegression,0.036,0.054,0.036,0.018,-0.126,0.339
1,Hialeah,1,RandomForest,0.062,0.000,0.046,0.000,-0.142,0.058
2,Hialeah,2,LogisticRegression,-0.018,0.018,-0.008,-0.045,-0.066,0.475
3,Hialeah,2,RandomForest,0.036,0.018,0.030,0.013,-0.032,0.329
4,Hialeah,4,LogisticRegression,0.046,-0.023,0.065,-0.090,-0.052,0.115
5,Hialeah,4,RandomForest,0.123,-0.015,0.169,-0.012,-0.067,0.200
6,Miami,1,LogisticRegression,0.031,-0.003,0.117,-0.049,0.031,0.179
7,Miami,1,RandomForest,0.031,0.014,0.096,-0.060,-0.016,0.133
8,Miami,2,LogisticRegression,0.034,0.016,0.068,0.022,0.069,0.108
9,Miami,2,RandomForest,0.021,0.024,0.063,0.030,-0.086,-0.006


In [12]:
manifest={
    "input_directory":str(INPUT_DIR),
    "source_name_from_notebook1":contract["source_file"],
    "source_sha256_from_notebook1":contract["source_sha256"],
    "forecast_horizon_unit":contract["forecast_horizon_unit"],
    "cities":list(CITIES),"horizons_periods":list(HORIZONS),
    "peak_threshold_definition":"0.80 quantile of TRAIN ORIGIN period mosquito counts per city/horizon; fixed for test targets",
    "split":"first 80% of eligible origin periods as candidate training; remaining 20% test; purge training target periods >= first test origin period",
    "test_fraction":TEST_FRACTION,
    "probability_decision_threshold":DECISION_THRESHOLD,
    "algorithm_hyperparameters":{
        "LogisticRegression":"median imputation, scaling, class_weight=balanced, lbfgs, max_iter=3000",
        "RandomForest":"median imputation, 400 trees, min_samples_leaf=5, max_features=sqrt, class_weight=balanced_subsample",
    },
    "environment_features":ENVIRONMENT_FEATURES,
    "history_features":HISTORY_FEATURES,
    "statistical_testing":"none; between-feature-set differences descriptive",
    "model_selection":"none; prespecified algorithms/hyperparameters; no tuning with test outcomes",
    "notes":["Source reporting periods may be 1–7 days rather than standard weeks.",
             "Observations are unstandardized aggregated collections and may vary with trap effort.",
             "Environmental associations and the lagged city-period features inherit source-construction limitations.",
             "The existing manuscript's Sunday-ending weekly analyses are a separate dataset."],
    "software":{"python":platform.python_version(),"pandas":pd.__version__,"sklearn":sklearn.__version__,"numpy":np.__version__},
    "number_models":int(len(metrics)),
    "number_predictions":int(len(predictions)),
}
(OUTPUT_DIR/"ablation_run_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Results files:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):print(" ",p.name)
print("Saved manifest and figures.")

Results files:
  ablation_descriptive_deltas.csv
  ablation_performance_all.csv
  ablation_predictions_all.csv
  chronological_split_audit.csv
  excluded_training_features.csv
  publication_ablation_table.csv
Saved manifest and figures.
